# 구내식당 식수 인원 예측: 업그레이드 모델 (upmodel5)
## 전략: 참여율(Ratio) + 모델 앙상블(XGB, LGBM, CAT)
1. **참여율 전략**: `식사계 / 식사가능자수`를 타겟으로 예측
2. **앙상블 기법**: XGBoost, LightGBM, CatBoost를 학습시킨 후 산술 평균(Simple Averaging) 적용
3. **피처**: 기본 시간/인원 피처 활용

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. 데이터 로드
train = pd.read_csv('data/train_median.csv')
test = pd.read_csv('data/test.csv')
submission = pd.read_csv('data/sample_submission.csv')

def preprocess_basic(df):
    df = df.copy()
    df['일자'] = pd.to_datetime(df['일자'])
    df['month'] = df['일자'].dt.month
    df['day'] = df['일자'].dt.day
    df['weekday'] = df['일자'].dt.weekday
    df['식사가능자수'] = df['본사정원수'] - (df['본사휴가자수'] + df['본사출장자수'] + df['현본사소속재택근무자수'])
    return df

train_df = preprocess_basic(train)
test_df = preprocess_basic(test)

train_df['target_lunch'] = train_df['중식계'] / train_df['식사가능자수']
train_df['target_dinner'] = train_df['석식계'] / train_df['식사가능자수']

In [ ]:
# 2. 모델 정의
features = ['month', 'day', 'weekday', '식사가능자수', '본사출장자수', '본사시간외근무명령서승인건수']

xgb_params = {'n_estimators': 1000, 'learning_rate': 0.03, 'max_depth': 6, 'random_state': 42}
lgb_params = {'n_estimators': 1000, 'learning_rate': 0.03, 'max_depth': 6, 'random_state': 42, 'verbosity': -1}
cat_params = {'n_estimators': 1000, 'learning_rate': 0.03, 'depth': 6, 'random_state': 42, 'silent': True}

def get_ensemble_pred(X_train, y_train, X_test):
    m1 = XGBRegressor(**xgb_params)
    m2 = LGBMRegressor(**lgb_params)
    m3 = CatBoostRegressor(**cat_params)
    
    m1.fit(X_train, y_train)
    m2.fit(X_train, y_train)
    m3.fit(X_train, y_train)
    
    p1 = m1.predict(X_test)
    p2 = m2.predict(X_test)
    p3 = m3.predict(X_test)
    
    return (p1 + p2 + p3) / 3

print("중식 앙상블 학습 중...")
pred_l_ratio = get_ensemble_pred(train_df[features], train_df['target_lunch'], test_df[features])
print("석식 앙상블 학습 중...")
pred_d_ratio = get_ensemble_pred(train_df[features], train_df['target_dinner'], test_df[features])

In [ ]:
# 3. 결과 복원 및 저장
submission['중식계'] = pred_l_ratio * test_df['식사가능자수']
submission['석식계'] = pred_d_ratio * test_df['식사가능자수']
submission.to_csv('upmodel5_submission.csv', index=False)
print("upmodel5: 참여율 + 앙상블 모델 완료")